# 4.1 手写回测引擎

## 学习目标
- 从零构建一个简单的事件驱动回测框架
- 实现双均线（SMA）交叉策略的完整回测
- 理解仓位管理、手续费扣除与绩效统计

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.rcParams['figure.figsize'] = (13, 5)

# 下载数据
data = yf.download('SPY', start='2018-01-01', end='2024-01-01', progress=False)
close = data['Close'].squeeze()
print('数据准备完成 ✅')

## 1. 生成交易信号

In [ ]:
# 双均线策略参数
FAST = 20
SLOW = 60

signals = pd.DataFrame(index=close.index)
signals['price'] = close
signals['sma_fast'] = close.rolling(FAST).mean()
signals['sma_slow'] = close.rolling(SLOW).mean()

# ⚠️  关键：用 shift(1) 避免 Look-ahead Bias！
# 信号在当天生成，第二天开盘执行
signals['signal'] = 0
signals.loc[signals['sma_fast'] > signals['sma_slow'], 'signal'] = 1   # 多头
signals.loc[signals['sma_fast'] < signals['sma_slow'], 'signal'] = -1  # 空仓

# 持仓变化（只在信号切换时交易）
signals['position'] = signals['signal'].shift(1)  # 次日持仓
signals['trade'] = signals['position'].diff().fillna(0)  # 交易发生时刻

n_trades = (signals['trade'] != 0).sum()
print(f'总交易次数: {n_trades}')
signals[['price', 'sma_fast', 'sma_slow', 'signal', 'position']].tail(5)

## 2. 计算每日收益率

In [ ]:
COMMISSION = 0.001  # 0.1% 手续费（单边）

daily_ret = close.pct_change()
# 策略收益 = 持仓方向 × 市场收益
signals['strategy_ret'] = signals['position'] * daily_ret

# 扣除手续费（发生交易时）
trade_cost = signals['trade'].abs() * COMMISSION
signals['strategy_ret'] -= trade_cost
signals['market_ret'] = daily_ret

# 累积收益
signals = signals.dropna()
signals['cum_strategy'] = (1 + signals['strategy_ret']).cumprod()
signals['cum_market'] = (1 + signals['market_ret']).cumprod()

signals[['cum_strategy', 'cum_market']].tail(3)

## 3. 可视化结果

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

# 价格 + 均线 + 买卖点
ax1.plot(signals.index, signals['price'], linewidth=1, label='SPY', alpha=0.7)
ax1.plot(signals.index, signals['sma_fast'], label=f'SMA{FAST}', linewidth=1.3)
ax1.plot(signals.index, signals['sma_slow'], label=f'SMA{SLOW}', linewidth=1.3)
buy_signals = signals[signals['trade'] > 0]
sell_signals = signals[signals['trade'] < 0]
ax1.scatter(buy_signals.index, buy_signals['price'],
             marker='^', color='green', s=80, zorder=5, label='买入')
ax1.scatter(sell_signals.index, sell_signals['price'],
             marker='v', color='red', s=80, zorder=5, label='卖出')
ax1.set_title(f'双均线策略 (SMA{FAST}/{SLOW}) | SPY', fontsize=13)
ax1.legend(ncol=3)
ax1.grid(alpha=0.3)

# 累积收益对比
ax2.plot(signals.index, signals['cum_strategy'], label='策略收益', linewidth=1.5, color='steelblue')
ax2.plot(signals.index, signals['cum_market'], label='买入持有', linewidth=1.5,
          color='orange', linestyle='--')
ax2.set_title('累积收益对比', fontsize=13)
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. 绩效指标汇总

In [ ]:
def performance_summary(ret_series, name='策略'):
    total_ret = (1 + ret_series).prod() - 1
    n_days = len(ret_series)
    annual_ret = (1 + total_ret) ** (252 / n_days) - 1
    annual_vol = ret_series.std() * np.sqrt(252)
    sharpe = annual_ret / annual_vol
    cum = (1 + ret_series).cumprod()
    rolling_max = cum.cummax()
    mdd = ((cum - rolling_max) / rolling_max).min()
    calmar = annual_ret / abs(mdd)
    return pd.Series({
        '总收益率': f'{total_ret:.2%}',
        '年化收益率': f'{annual_ret:.2%}',
        '年化波动率': f'{annual_vol:.2%}',
        '夏普比率': f'{sharpe:.2f}',
        '最大回撤': f'{mdd:.2%}',
        'Calmar 比率': f'{calmar:.2f}',
    }, name=name)

strat = performance_summary(signals['strategy_ret'], '双均线策略')
mkt = performance_summary(signals['market_ret'], 'Buy & Hold')
pd.DataFrame([strat, mkt]).T

## 🎯 练习

1. 更改均线参数 `FAST=5, SLOW=20`，对比结果。
2. 增加手续费至 `0.005`（0.5%），观察策略表现如何变化。这揭示了什么？
3. 将策略从 SPY 换成波动更大的 TSLA，结果是否更好或更差？

---
**下一节** → `02_backtrader_intro.ipynb`